# SAMPIC external-trigger diagnostics

Inspect `SG` trigger groups, optional `ST` timing, and `SD` hits. Periodic and ordinary self-trigger runs usually have no SG records; this is reported as not applicable. `SV` vendor trigger records are counted separately and are not mistaken for hit associations.


In [ ]:
# Edit this path, then run the notebook from the top.
# Absolute paths and ~ are supported; relative paths start at the app directory.
DATA_FILE = ".artifacts/bank_format_verification/run00266/output.root"


In [ ]:
from pathlib import Path
import sys, os
# Works when Jupyter starts in the repository or a notebook subdirectory.
APP_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                if (p / 'notebooks/notebook_support.py').is_file())
if str(APP_ROOT / 'notebooks') not in sys.path:
    sys.path.insert(0, str(APP_ROOT / 'notebooks'))
from notebook_support import (open_data, present, hit_waveform, hit_baseline,
    hit_amplitude, hit_peak, waveform_entries as find_waveform_entries,
    sampic_unit, integer, waveform_packets, sampic_event_bytes)
import ROOT
r = ROOT
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
root_file, tree, DATA_FILE = open_data('SAMPIC', DATA_FILE)
f = file = root_file
plt.rcParams.update({'figure.figsize': (12, 5), 'axes.grid': True, 'grid.alpha': 0.25})
UNIT = sampic_unit(tree)
print('Waveform display units:', UNIT)


In [ ]:
records = []
vendor_trigger_count = 0
for idx in range(tree.GetEntries()):
    tree.GetEntry(idx)
    if present(tree, 'trigger_records'):
        vendor_trigger_count += len(tree.sampic_trigger_records.triggers)
    if not present(tree, 'trigger_group'): continue
    tg = tree.sampic_trigger_group
    records.append(dict(entry=idx, fpga_id=int(tg.fpga_trigger_id), external_id=int(tg.external_trigger_id),
        trigger_index=int(tg.trigger_index_in_sampic_event), assigned_hits=int(tg.assigned_hits),
        ambiguous_hits=int(tg.ambiguous_hits), trigger_ns=float(tg.trigger_timestamp_ns),
        hit_reference_ns=float(tg.hit_reference_timestamp_ns),
        timing_nhits=int(tree.sampic_event_timing.nhits) if present(tree, 'event_timing') else None,
        sd_nhits=len(tree.sampic_event.hits) if present(tree, 'event') else None))
print(f'SG records: {len(records)}; SV vendor trigger records: {vendor_trigger_count}')
if not records: print('No SG records: external-trigger association diagnostics are not applicable to this run.')


In [ ]:
# Counter modulus is configurable; 256 matches the vendor FPGA trigger counter.
FPGA_ID_MODULUS = 256
if records:
    ids = np.array([x['fpga_id'] for x in records], dtype=np.int64)
    times = np.array([x['trigger_ns'] for x in records])
    intervals = np.diff(times)
    steps = np.diff(ids) % FPGA_ID_MODULUS
    print('Non-unit FPGA ID steps:', np.count_nonzero(steps != 1))
    positive = intervals[intervals > 0]
    if positive.size: print(f'Mean positive-interval rate: {1e9/positive.mean():.3f} Hz')
    print('Nonpositive timestamp intervals:', np.count_nonzero(intervals <= 0))
    comparisons = [x for x in records if x['timing_nhits'] is not None]
    print('SG/ST comparisons:', len(comparisons), 'mismatches:', sum(x['assigned_hits'] != x['timing_nhits'] for x in comparisons))
    comparisons_sd = [x for x in records if x['sd_nhits'] is not None]
    print('SG/SD comparisons:', len(comparisons_sd), 'mismatches:', sum(x['assigned_hits'] != x['sd_nhits'] for x in comparisons_sd))
    fig, axes = plt.subplots(2,2,figsize=(12,8))
    axes[0,0].plot([x['entry'] for x in records], ids, '.'); axes[0,0].set(xlabel='Tree entry', ylabel='FPGA ID')
    axes[0,1].hist(intervals*1e-6, bins=40); axes[0,1].set(xlabel='Trigger interval [ms]', ylabel='Pairs')
    axes[1,0].hist([x['assigned_hits'] for x in records], bins=30); axes[1,0].set(xlabel='Assigned hits', ylabel='Triggers')
    with_hits = [x for x in records if x['assigned_hits'] > 0]
    axes[1,1].plot([x['entry'] for x in with_hits], [x['hit_reference_ns']-x['trigger_ns'] for x in with_hits], '.')
    axes[1,1].set(xlabel='Tree entry', ylabel='Hit reference − trigger [ns]')
    plt.tight_layout(); plt.show()
    print('Ambiguous hits:', sum(x['ambiguous_hits'] for x in records))
